# 04. 選択的上書き - 特定の範囲だけを入れ替える

日次バッチでよくある状況を考えます。

「9月11日のデータに間違いがあったので、その日の分だけ作り直したい」

素直にやろうとすると、どれも具合が悪いことに気づきます。

| やり方 | 問題 |
|---|---|
| `INSERT` で追加する | 元のデータが残ったままなので、その日の分が二重になる |
| テーブル全体を上書きする | 他の日のデータまで消える |
| `DELETE` してから `INSERT` する | 2段階になる。途中で落ちるとデータが消えたままになる |

この「特定の範囲だけを、まとめて入れ替える」やり方を扱います。
Databricksには方法が2つあり、動きがかなり違います。

1. `replaceWhere` … **人間が範囲を条件で宣言する**
2. `REPLACE USING` … **キー列の一致でDeltaが行ごとに判断する**

このノートブックで確かめること:

1. Liquid Clustering とパーティションの違い
2. `replaceWhere` で範囲を入れ替える。何度実行しても同じ結果になること
3. 条件と書き込むデータが食い違うとどうなるか
4. **ソースが空だったとき**、2つの方法で結果が変わること

**前提**: `00_setup` を実行済みであること。

## 準備

In [3]:
from datetime import date

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [4]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.silver.daily_orders"

# 行を作るときに毎回書くので、列の定義をまとめておく
SCHEMA = "order_id INT, product STRING, amount INT, order_date DATE"

## 1. テーブルのレイアウト - パーティションと Liquid Clustering

「9月11日のデータ」を扱うとき、テーブル全体を読まずに済ませたい。
そのためにデータの置き方を工夫します。方法が2つあります。

**パーティション** は、列の値ごとにフォルダを物理的に分ける方式です。
`order_date` で切ると、日付ごとに別フォルダにファイルが置かれます。長く使われてきた方法です。

ただし弱点があります。

- **後から変えられない**。切り直すにはテーブル全体を書き直すことになる
- 値の種類が多すぎると、小さいファイルが大量にできて逆に遅くなる
- 逆に少なすぎると、絞り込みが効かない

**Liquid Clustering** は、フォルダを分けずに、ファイル内のデータの並びを整える方式です。
どのファイルにどの範囲の値が入っているかを統計情報として持ち、読むときに不要なファイルを飛ばします。

現在のDatabricksは「すべての新しいテーブルで Liquid Clustering を推奨」としています。
**後からキーを変えられる**のが大きな利点で、既存データを書き直さずに変更できます。

ここでは推奨に従って Liquid Clustering で作ります。詳しくは `10` で扱います。

In [5]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# DDL
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        order_date DATE
    )
    CLUSTER BY (order_date)
""")

# DML
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000, DATE '2026-09-10'),
        (2, 'monitor',   40000, DATE '2026-09-10'),
        (3, 'keyboard',  12000, DATE '2026-09-11'),
        (4, 'mouse',      5000, DATE '2026-09-11'),
        (5, 'headset',   20000, DATE '2026-09-12')
""")

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,12000,2026-09-11
3,4,mouse,5000,2026-09-11
4,5,headset,20000,2026-09-12


In [7]:
# パーティションではなくクラスタリングキーが設定されていることを確認する
display(spark.sql(f"DESCRIBE DETAIL {TABLE}").select("partitionColumns", "clusteringColumns", "numFiles"))

,partitionColumns,clusteringColumns,numFiles
0,[],[order_date],1


## 2. 素朴に追加するとどうなるか

9月11日のデータを作り直したい、という状況です。まず `INSERT` で入れてみます。

In [ ]:
fixed_rows = spark.createDataFrame(
    [
        (3, "keyboard", 13000, date(2026, 9, 11)),
        (4, "mouse", 5500, date(2026, 9, 11)),
    ],
    SCHEMA,
)

(
    fixed_rows.write.format("delta")
    .mode("append")  # append で書き込むと、同じ order_id の行が複数できてしまう
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).where("order_date = DATE '2026-09-11'").orderBy("order_id"))

,order_id,product,amount,order_date
0,3,keyboard,12000,2026-09-11
1,3,keyboard,13000,2026-09-11
2,4,mouse,5000,2026-09-11
3,4,mouse,5500,2026-09-11


古い行と新しい行が両方残り、9月11日のデータが二重になりました。

かといってテーブル全体を上書き (`mode("overwrite")`) すると、
触るつもりのなかった9月10日と12日まで消えます。

## 3. `replaceWhere` で入れ替える

まず、二重になった状態を元に戻します。

In [9]:
spark.sql(f"DELETE FROM {TABLE} WHERE order_date = DATE '2026-09-11'")
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (3, 'keyboard', 12000, DATE '2026-09-11'),
        (4, 'mouse',     5000, DATE '2026-09-11')
""")

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,12000,2026-09-11
3,4,mouse,5000,2026-09-11
4,5,headset,20000,2026-09-12


`mode("overwrite")` に `replaceWhere` を添えると、上書きする範囲を条件で絞れます。

```python
.mode("overwrite")  # 上書きする
.option("replaceWhere", "order_date = '2026-09-11'")  # ただしこの条件に合う範囲だけ
```

条件に合う既存の行が消え、書き込むデータが入ります。
この2つは **1回の操作としてまとめて行われます**。途中の中途半端な状態が他から見えることはありません。

条件はパーティション列である必要はありません。テーブルの任意の列で書けます。
昔はパーティション列だけという制限がありましたが、今は外れています。

条件は **真偽値を返す式** でなければなりません。列名だけを渡すことはできません。
`"order_date"` と書いても日付が返るだけで条件にならないため、エラーになります。

In [11]:
display(fixed_rows)

(
    fixed_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")  # ただしこの条件に合う範囲だけoverwrite
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,3,keyboard,13000,2026-09-11
1,4,mouse,5500,2026-09-11


,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,13000,2026-09-11
3,4,mouse,5500,2026-09-11
4,5,headset,20000,2026-09-12


## 4. 何度実行しても同じか

日次バッチでは、同じ処理が2回動いてしまうことがあります。
ジョブのリトライ、手動での再実行、上流からの再送などです。

さきほどとまったく同じ書き込みをもう一度実行します。件数がどうなるか予想してください。

In [17]:
(
    fixed_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")
    .saveAsTable(TABLE)
)

# order_date ごとに件数を集計して、2026-09-11 の件数が 2 件になっていることを確認する
display(spark.sql(f"""
    SELECT order_date, count(*) AS cnt
    FROM {TABLE}
    GROUP BY order_date
    ORDER BY order_date
"""))

,order_date,cnt
0,2026-09-10,2
1,2026-09-11,2
2,2026-09-12,1


## 5. 条件と書き込むデータが食い違うと

`replaceWhere` には安全装置があります。
「9月11日を入れ替える」と宣言したのに、9月12日のデータが混ざっていたらどうなるでしょうか。

In [14]:
wrong_rows = spark.createDataFrame(
    [
        (3, "keyboard", 13000, date(2026, 9, 11)),
        (99, "cable", 1000, date(2026, 9, 12)),  # 宣言した範囲の外
    ],
    SCHEMA,
)

try:
    (
        wrong_rows.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", "order_date = '2026-09-11'")
        .saveAsTable(TABLE)
    )
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

AnalysisException
[DELTA_REPLACE_WHERE_MISMATCH] Written data does not conform to partial table overwrite condition or constraint 'order_date = '2026-09-11''.
[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint EXPRESSION(('order_date = 2026-09-11)) (order_date = '2026-09-11') violated by row with values:
 - order_date : 2026-09-12.


JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	


拒否されたはずです。

もしこれが通ってしまうと、「9月11日を入れ替えたつもりが、9月12日に知らないデータが増えていた」
という事故が起きます。宣言した範囲と実際に書くデータが一致していることを、Deltaが確認してくれています。

## 6. ソースが空だったら

ここが `replaceWhere` を使う上で一番気をつけるところです。

上流の処理に不具合があり、9月11日のデータが1件も取れなかったとします。
その空の結果を、いつもどおり `replaceWhere` で書き込むとどうなるでしょうか。

予想してから実行してください。

In [21]:
# 空の DataFrame を作って、replaceWhere で上書きしようとする
empty_rows = spark.createDataFrame([], SCHEMA)

(
    empty_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")  # ← 条件は日付指定
    .saveAsTable(TABLE)
)

# あるはずの 2026-09-11 の行が消えてしまったことを確認する
display(spark.sql(f"SELECT * FROM {TABLE}"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,5,headset,20000,2026-09-12


9月11日が消えました。

`replaceWhere` は「この範囲を、これから書く内容で置き換える」という指示です。
書く内容が空なら、範囲は空になります。指示どおりの動作です。

これは **意図どおりの場面もあれば、事故の場面もあります**。

- 上流で本当に取り消しが起きて0件になった → 消えるのが正しい
- 上流の障害でたまたま0件だった → 消えてはいけない

`replaceWhere` はこの2つを区別できません。区別するには、**書き込む前に自分で件数を確認** して
処理を止めるなどの作りが必要になります。

## 7. `replaceUsing` - 行ごとに判断させる

もう1つの方法です。考え方が根本的に違います。

`replaceWhere` が「範囲を人間が宣言する」のに対し、
`replaceUsing` は **キー列の値が一致する行を Delta が探して置き換えます**。
範囲の宣言は要りません。

```python
.option("replaceUsing", "キー列, キー列, ...")
```

SQLでは `INSERT INTO TABLE ... REPLACE USING (...)` と書きます。どちらでも同じです。

まず9月11日のデータを戻します。

In [22]:
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (3, 'keyboard', 12000, DATE '2026-09-11'),
        (4, 'mouse',     5000, DATE '2026-09-11')
""")

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,12000,2026-09-11
3,4,mouse,5000,2026-09-11
4,5,headset,20000,2026-09-12


In [ ]:
(
    fixed_rows.write.format("delta")  # 2026-09-11 の行が含まれるdf
    .mode("overwrite")
    .option("replaceUsing", "order_id, order_date")  # 条件ではなくキー列を指定する
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,13000,2026-09-11
3,4,mouse,5500,2026-09-11
4,5,headset,20000,2026-09-12


## 8. `REPLACE USING` でソースが空だったら

`6.` とまったく同じ状況を、今度は `REPLACE USING` で試します。
結果がどうなるか予想してください。

In [27]:
(
    empty_rows.write.format("delta")  # 空の DataFrame
    .mode("overwrite")
    .option("replaceUsing", "order_id, order_date")
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).orderBy("order_date", "order_id"))

,order_id,product,amount,order_date
0,1,laptop,150000,2026-09-10
1,2,monitor,40000,2026-09-10
2,3,keyboard,13000,2026-09-11
3,4,mouse,5500,2026-09-11
4,5,headset,20000,2026-09-12


`replaceUsing` は、ソースに存在する行のキーを見て、一致する行を置き換えます。
ソースが空なら一致する行が1つもないので、**何も起きません**。データは消えません。

これが2つの方法の決定的な違いです。

### 複雑な条件* で絞りたい場合

複雑な条件を自分で書きたいときは `replaceOn` を使います。

```python
(
    source_df.alias("s").write
    .mode("overwrite")
    .option("targetAlias", "t")
    .option("replaceOn", "s.order_id <=> t.order_id AND s.order_date <=> t.order_date")
    .saveAsTable(TABLE)
)
```
* 複雑な条件とは：`<=>` 演算子など  
> `<=>` は **NULL同士も一致とみなす** 等価演算子です。`=` では `NULL = NULL` が `NULL` になってしまい、
> 一致と判定されません。キーにNULLが入りうる場合は `replaceOn` が必要になります。

ドキュメントには、「複雑な条件」とは書かれていますが、不等号や範囲条件、計算式が使えるかは明記されていません。

### 補足: `partitionOverwriteMode`

「条件を書かず、書き込むデータに含まれる範囲だけを上書きする」機能は、実は以前からありました。

```python
.option("partitionOverwriteMode", "dynamic")
```

書き込むデータに9月11日しか入っていなければ、9月11日のパーティションだけが上書きされます。
`replaceUsing` と発想は同じです。

ただし名前のとおり **パーティションに依存** しています。Liquid Clustering のテーブルには
パーティションが無いので使えません。
ドキュメントも「可能な場合は `REPLACE USING` を使うこと」としています。
これをレイアウトに依存しない形に作り直したものが `replaceUsing` だと考えると、位置づけが分かりやすいです。

## 9. 使い分け

| | 範囲の決め方 | 対応レイアウト | ソースが空のとき |
|---|---|---|---|
| `replaceWhere` | 条件文を人間が書く | パーティション / 非パーティション | **範囲の行が消える** |
| `partitionOverwriteMode=dynamic` | 書くデータから自動判定 | **パーティションのみ** | 対象パーティションが無いので何も起きない |
| `replaceUsing` | キー列を指定 | パーティション / 非パーティション / **Liquid Clustering** | **何も起きない** |

`replaceWhere` は宣言と違うデータを書こうとするとエラーになりますが、`replaceUsing` は行ごとの判断なので
そうした安全装置はありません。ソースに入っていたものがそのまま反映されます。

どちらが良いかは、**消えてほしいかどうか** で決まります。

「9月11日はこの内容が全てである」と言い切れる処理なら `replaceWhere` が向きます。
上流から消えた行があれば、こちらでも消えるのが正しい挙動です。

「届いた分を反映したい。届かなかったものには触らない」なら `replaceUsing` です。
部分的な更新が繰り返し届くような処理に向きます。

## 考えてみる

- `01` のチェックポイントによる重複防止と、`replaceWhere` による重複防止は何が違うのでしょうか
- 上流の障害で0件になったときにデータを消したくない場合、`replaceWhere` を使うならどう作りますか
- Liquid Clustering のキーを後から変更できるのは、どういう仕組みだからでしょうか

### 答え

**Q1. チェックポイントによる重複防止との違い**

守っている場所が違います。

- チェックポイント … **読む側** が「どこまで読んだか」を覚えている。同じ入力を2度読まない
- `replaceWhere` … **書く側** が「どの範囲を担当するか」を宣言する。同じ範囲を何度書いても結果は1つ

チェックポイントは、上流から同じデータが再送されてきた場合には無力です。読む側にとっては
新しいファイルなので取り込んでしまいます。
一方 `replaceWhere` は、何が来ようとその範囲を宣言した内容で置き換えるので、再送に強くなります。

**Q2. 0件のときに消したくない場合**

`replaceWhere` 自体には区別する手段がないので、**書き込む前に自分で判断します**。

```python
if source_df.isEmpty():
    raise ValueError("上流が0件。異常の可能性があるため中断する")
```

「0件はありえない」と言える処理なら、これで事故を防げます。
一方「0件もありうる」処理では、どちらが正しいか処理側では判断できません。
上流に「意図的な0件」と「取得失敗」を区別できる情報を持たせるか、`REPLACE USING` を使うことになります。

**Q3. Liquid Clustering のキーを後から変えられる理由**

パーティションは、値ごとにフォルダを分ける **物理的な配置** です。
キーを変えるとフォルダ構成そのものが変わるため、全データを置き直すことになります。

Liquid Clustering はフォルダを分けません。データの並べ方と統計情報の問題なので、
キーを変えても既存のファイルはそのまま置いておけます。
変更後に書かれるデータから新しいキーで並び、古いデータは最適化の過程で徐々に整理されていきます。

「配置を決め打ちしない」という設計が、後から変えられることに繋がっています。

## 後片付け

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")